# DX 704 Week 6 Project

This project will develop a treatment plan for a fictious illness "Twizzleflu".
Twizzleflu is a mild illness caused by a virus.
The main symptoms are a mild fever, fidgeting, and kicking the blankets off the bed or couch.
Mild dehydration has also been reported in more severe cases.
These symptoms typically last 1-2 weeks without treatment.
Word on the internet says that Twizzleflu can be cured faster by drinking copious orange juice, but this has not been supported by evidence so far.
You will be provided with a theoretical model of Twizzleflu modeled as a Markov decision process.
Based on the model, you will compute optimal treatment plans to optimize different criteria, and compare patient discomfort with the different plans.

The full project description, a template notebook, and raw data are available on GitHub: [Project 6 Materials](https://github.com/bu-cds-dx704/dx704-project-06).

We will model Twizzleflu as a Markov decision process.
The model transition probabilities are provided in the file "twizzleflu-transitions.tsv" and the expected rewards are in "twizzleflu-rewards.tsv".
The goal for Twizzleflu is to minimize the expected discomfort of the patient which is expressed as negative rewards in the file.

## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Evaluate a Do Nothing Plan

One of the treatment actions is to do nothing.
Calculate the expected discomfort (not rewards) of a policy that always does nothing.

Hint: for this value calculation and later ones, use value iteration.
The analytical solution has difficulties in practice when there is no discount factor.

In [6]:
# YOUR CHANGES HERE
import pandas as pd
import numpy as np

transitions = pd.read_csv("twizzleflu-transitions.tsv", sep="\t")
rewards = pd.read_csv("twizzleflu-rewards.tsv", sep="\t")

print("Available actions:", rewards["action"].unique())
print("Available states:", rewards["state"].unique())

do_nothing = next(
    a for a in rewards["action"].unique()
    if "nothing" in str(a).lower()
)

T = transitions[transitions["action"] == do_nothing]
R = rewards[rewards["action"] == do_nothing]

states = sorted(
    set(transitions["state"]) |
    set(transitions["next_state"])
)

V = {s: 0.0 for s in states}
reward_map = dict(zip(R["state"], R["reward"]))

tolerance = 1e-10

for iteration in range(100000):
    new_V = V.copy()

    for s in states:
        if s not in reward_map:
            continue

        future_value = sum(
            row.probability * V[row.next_state]
            for row in T[T["state"] == s].itertuples()
        )

        new_V[s] = reward_map[s] + future_value

    delta = max(abs(new_V[s] - V[s]) for s in states)
    V = new_V

    if delta < tolerance:
        break

discomfort = {s: -v for s, v in V.items()}

print("\nExpected discomfort by state:")
for s, d in discomfort.items():
    print(f"{s}: {d:.6f}")

print(f"\nIterations: {iteration + 1}")
print(f"Converged: {delta < tolerance}")


Available actions: <StringArray>
['do-nothing', 'drink-oj', 'sleep-8']
Length: 3, dtype: str
Available states: <StringArray>
[ 'exposed-1',  'exposed-2',  'exposed-3', 'symptoms-1', 'symptoms-2',
 'symptoms-3',  'recovered']
Length: 7, dtype: str

Expected discomfort by state:
exposed-1: 3.413333
exposed-2: 4.266667
exposed-3: 5.333333
recovered: -0.000000
symptoms-1: 6.666667
symptoms-2: 5.000000
symptoms-3: 1.666667

Iterations: 83
Converged: True


Save the expected discomfort by state to a file "do-nothing-discomfort.tsv" with columns state and expected_discomfort.

In [7]:
# YOUR CHANGES HERE


results = pd.DataFrame({
    "state": list(discomfort.keys()),
    "discomfort": list(discomfort.values())
})

results.to_csv(
    "do-nothing-discomfort.tsv",
    sep="\t",
    index=False
)

print(results)


        state  discomfort
0   exposed-1    3.413333
1   exposed-2    4.266667
2   exposed-3    5.333333
3   recovered   -0.000000
4  symptoms-1    6.666667
5  symptoms-2    5.000000
6  symptoms-3    1.666667


Submit "do-nothing-discomfort.tsv" in Gradescope.

## Part 2: Compute an Optimal Treatment Plan

Compute an optimal treatment plan for Twizzleflu.
It should minimize the expected discomfort (maximize the rewards).

In [8]:
# YOUR CHANGES HERE

states = sorted(
    set(transitions["state"]) |
    set(transitions["next_state"])
)

actions = list(rewards["action"].unique())

reward_map = {
    (row.state, row.action): row.reward
    for row in rewards.itertuples(index=False)
}


transition_map = {}

for row in transitions.itertuples(index=False):
    key = (row.state, row.action)

    transition_map.setdefault(key, []).append(
        (row.next_state, row.probability)
    )

V_opt = {s: 0.0 for s in states}
optimal_policy = {}

tolerance = 1e-10
max_iterations = 100000

# Optimal value iteration
for iteration in range(max_iterations):
    new_V = V_opt.copy()

    for s in states:
        action_values = {}

        for a in actions:
            key = (s, a)

            if key not in reward_map:
                continue

            future_value = sum(
                p * V_opt[next_state]
                for next_state, p in transition_map.get(key, [])
            )

            action_values[a] = reward_map[key] + future_value

        if action_values:
            best_action = max(action_values, key=action_values.get)

            new_V[s] = action_values[best_action]
            optimal_policy[s] = best_action

    delta = max(
        abs(new_V[s] - V_opt[s])
        for s in states
    )

    V_opt = new_V

    if delta < tolerance:
        break

for s in states:
    action_values = {}

    for a in actions:
        key = (s, a)

        if key not in reward_map:
            continue

        action_values[a] = reward_map[key] + sum(
            p * V_opt[next_state]
            for next_state, p in transition_map.get(key, [])
        )

    if action_values:
        optimal_policy[s] = max(
            action_values, key=action_values.get
        )


results = pd.DataFrame({
    "state": states,
    "optimal_action": [
        optimal_policy.get(s, "terminal")
        for s in states
    ],
    "expected_discomfort": [
        -V_opt[s] for s in states
    ]
})

print(results.to_string(index=False))
print(f"\nIterations: {iteration + 1}")
print(f"Converged: {delta < tolerance}")



     state optimal_action  expected_discomfort
 exposed-1        sleep-8                 0.75
 exposed-2        sleep-8                 1.50
 exposed-3        sleep-8                 3.00
 recovered     do-nothing                -0.00
symptoms-1       drink-oj                 6.00
symptoms-2       drink-oj                 4.50
symptoms-3       drink-oj                 1.50

Iterations: 100
Converged: True


Save the optimal actions for each state to a file "minimum-discomfort-actions.tsv" with columns state and action.

In [9]:
# YOUR CHANGES HERE

minimum_discomfort_actions = pd.DataFrame({
    "state": states,
    "action": [optimal_policy[s] for s in states]
})

minimum_discomfort_actions.to_csv(
    "minimum-discomfort-actions.tsv",
    sep="\t",
    index=False
)

print(minimum_discomfort_actions)



        state      action
0   exposed-1     sleep-8
1   exposed-2     sleep-8
2   exposed-3     sleep-8
3   recovered  do-nothing
4  symptoms-1    drink-oj
5  symptoms-2    drink-oj
6  symptoms-3    drink-oj


Submit "minimum-discomfort-actions.tsv" in Gradescope.

## Part 3: Expected Discomfort

Using your previous optimal policy, compute the expected discomfort for each state.

In [10]:
# YOUR CHANGES HERE

expected_discomfort = pd.DataFrame({
    "state": states,
    "discomfort": [-V_opt[s] for s in states]
})

expected_discomfort["discomfort"] = (
    expected_discomfort["discomfort"].clip(lower=0)
)

print(expected_discomfort.to_string(index=False))


     state  discomfort
 exposed-1        0.75
 exposed-2        1.50
 exposed-3        3.00
 recovered       -0.00
symptoms-1        6.00
symptoms-2        4.50
symptoms-3        1.50


Save your results in a file "minimum-discomfort-values.tsv" with columns state and expected_discomfort.

In [11]:
# YOUR CHANGES HERE


minimum_discomfort_values = pd.DataFrame({
    "state": states,
    "expected_discomfort": [
        max(0.0, -V_opt[s]) for s in states
    ]
})

minimum_discomfort_values.to_csv(
    "minimum-discomfort-values.tsv",
    sep="\t",
    index=False
)

print(minimum_discomfort_values.to_string(index=False))



     state  expected_discomfort
 exposed-1                 0.75
 exposed-2                 1.50
 exposed-3                 3.00
 recovered                 0.00
symptoms-1                 6.00
symptoms-2                 4.50
symptoms-3                 1.50


Submit "minimum-discomfort-values.tsv" in Gradescope.

## Part 4: Minimizing Twizzleflu Duration

Modifiy the Markov decision process to minimize the days until the Twizzle flu is over.
To do so, change the reward function to always be -1 if the current state corresponds to being sick (must have symptoms, exposed does not count) and 0 otherwise.
To be clear, the action does not matter for this reward function.


In [12]:
# YOUR CHANGES HERE
duration_rewards = rewards.copy()

duration_rewards["reward"] = duration_rewards["state"].apply(
    lambda s: -1.0 if s.startswith("symptoms") else 0.0
)

print(duration_rewards.to_string(index=False))


    action      state  reward
do-nothing  exposed-1     0.0
do-nothing  exposed-2     0.0
do-nothing  exposed-3     0.0
do-nothing symptoms-1    -1.0
do-nothing symptoms-2    -1.0
do-nothing symptoms-3    -1.0
do-nothing  recovered     0.0
  drink-oj  exposed-1     0.0
  drink-oj  exposed-2     0.0
  drink-oj  exposed-3     0.0
  drink-oj symptoms-1    -1.0
  drink-oj symptoms-2    -1.0
  drink-oj symptoms-3    -1.0
  drink-oj  recovered     0.0
   sleep-8  exposed-1     0.0
   sleep-8  exposed-2     0.0
   sleep-8  exposed-3     0.0
   sleep-8 symptoms-1    -1.0
   sleep-8 symptoms-2    -1.0
   sleep-8 symptoms-3    -1.0
   sleep-8  recovered     0.0


In [13]:

duration_reward_map = {
    (row.state, row.action): row.reward
    for row in duration_rewards.itertuples(index=False)
}
V_duration = {s: 0.0 for s in states}
duration_policy = {}

tolerance = 1e-10
max_iterations = 100000

for iteration in range(max_iterations):
    new_V = V_duration.copy()

    for s in states:
        action_values = {}

        for a in actions:
            key = (s, a)

            if key not in duration_reward_map:
                continue

            future_value = sum(
                p * V_duration[next_state]
                for next_state, p in transition_map.get(key, [])
            )

            action_values[a] = (
                duration_reward_map[key] + future_value
            )

        if action_values:
            best_action = max(
                action_values,
                key=action_values.get
            )

            new_V[s] = action_values[best_action]

    delta = max(
        abs(new_V[s] - V_duration[s])
        for s in states
    )

    V_duration = new_V

    if delta < tolerance:
        break

for s in states:
    action_values = {}

    for a in actions:
        key = (s, a)

        if key not in duration_reward_map:
            continue

        action_values[a] = (
            duration_reward_map[key]
            + sum(
                p * V_duration[next_state]
                for next_state, p in transition_map.get(key, [])
            )
        )

    if action_values:
        duration_policy[s] = max(
            action_values,
            key=action_values.get
        )

duration_results = pd.DataFrame({
    "state": states,
    "optimal_action": [
        duration_policy.get(s, "terminal")
        for s in states
    ],
    "expected_sick_days": [
        max(0.0, -V_duration[s])
        for s in states
    ]
})

print(duration_results.to_string(index=False))
print(f"\nIterations: {iteration + 1}")
print(f"Converged: {delta < tolerance}")


     state optimal_action  expected_sick_days
 exposed-1        sleep-8            1.250000
 exposed-2        sleep-8            2.500000
 exposed-3        sleep-8            5.000000
 recovered     do-nothing            0.000000
symptoms-1     do-nothing           10.000000
symptoms-2     do-nothing            6.666667
symptoms-3     do-nothing            3.333333

Iterations: 84
Converged: True


In [14]:

duration_rewards = rewards.copy()

duration_rewards["reward"] = duration_rewards["state"].apply(
    lambda s: -1.0 if s.startswith("symptoms") else 0.0
)

duration_rewards.to_csv(
    "duration-rewards.tsv",
    sep="\t",
    index=False
)

print(duration_rewards.to_string(index=False))


    action      state  reward
do-nothing  exposed-1     0.0
do-nothing  exposed-2     0.0
do-nothing  exposed-3     0.0
do-nothing symptoms-1    -1.0
do-nothing symptoms-2    -1.0
do-nothing symptoms-3    -1.0
do-nothing  recovered     0.0
  drink-oj  exposed-1     0.0
  drink-oj  exposed-2     0.0
  drink-oj  exposed-3     0.0
  drink-oj symptoms-1    -1.0
  drink-oj symptoms-2    -1.0
  drink-oj symptoms-3    -1.0
  drink-oj  recovered     0.0
   sleep-8  exposed-1     0.0
   sleep-8  exposed-2     0.0
   sleep-8  exposed-3     0.0
   sleep-8 symptoms-1    -1.0
   sleep-8 symptoms-2    -1.0
   sleep-8 symptoms-3    -1.0
   sleep-8  recovered     0.0


Save your new reward function in a file "duration-rewards.tsv" in the same format as "twizzleflu-rewards.tsv".

In [ ]:
# YOUR CHANGES HERE

...

Submit "duration-rewards.tsv" in Gradescope.

## Part 5: Optimize for Shorter Twizzleflu

Compute an optimal policy to minimize the duration of Twizzleflu.

In [15]:
# YOUR CHANGES HERE

duration_reward_map = {
    (row.state, row.action): row.reward
    for row in duration_rewards.itertuples(index=False)
}

V_duration = {s: 0.0 for s in states}
duration_policy = {}

tolerance = 1e-10
max_iterations = 100000


for iteration in range(max_iterations):
    new_V = V_duration.copy()

    for s in states:
        action_values = {}

        for a in actions:
            key = (s, a)

            if key not in duration_reward_map:
                continue

            future_value = sum(
                p * V_duration[next_state]
                for next_state, p in transition_map.get(key, [])
            )

            action_values[a] = (
                duration_reward_map[key] + future_value
            )

        if action_values:
            new_V[s] = max(action_values.values())

    delta = max(
        abs(new_V[s] - V_duration[s])
        for s in states
    )

    V_duration = new_V

    if delta < tolerance:
        break

for s in states:
    action_values = {}

    for a in actions:
        key = (s, a)

        if key not in duration_reward_map:
            continue

        action_values[a] = (
            duration_reward_map[key]
            + sum(
                p * V_duration[next_state]
                for next_state, p in transition_map.get(key, [])
            )
        )

    if action_values:
        duration_policy[s] = max(
            action_values,
            key=action_values.get
        )

duration_results = pd.DataFrame({
    "state": states,
    "optimal_action": [
        duration_policy.get(s, "terminal")
        for s in states
    ],
    "expected_sick_days": [
        max(0.0, -V_duration[s])
        for s in states
    ]
})

print(duration_results.to_string(index=False))
print(f"\nIterations: {iteration + 1}")
print(f"Converged: {delta < tolerance}")



     state optimal_action  expected_sick_days
 exposed-1        sleep-8            1.250000
 exposed-2        sleep-8            2.500000
 exposed-3        sleep-8            5.000000
 recovered     do-nothing            0.000000
symptoms-1     do-nothing           10.000000
symptoms-2     do-nothing            6.666667
symptoms-3     do-nothing            3.333333

Iterations: 84
Converged: True


In [16]:

policy_comparison = pd.DataFrame({
    "state": states,
    "minimum_discomfort_action": [
        optimal_policy.get(s, "terminal")
        for s in states
    ],
    "minimum_duration_action": [
        duration_policy.get(s, "terminal")
        for s in states
    ]
})

print(policy_comparison.to_string(index=False))


     state minimum_discomfort_action minimum_duration_action
 exposed-1                   sleep-8                 sleep-8
 exposed-2                   sleep-8                 sleep-8
 exposed-3                   sleep-8                 sleep-8
 recovered                do-nothing              do-nothing
symptoms-1                  drink-oj              do-nothing
symptoms-2                  drink-oj              do-nothing
symptoms-3                  drink-oj              do-nothing


Save the optimal actions for each state to a file "minimum-duration-actions.tsv" with columns state and action.

In [17]:
# YOUR CHANGES HERE

minimum_duration_actions = pd.DataFrame({
    "state": states,
    "action": [
        duration_policy.get(s, "do-nothing")
        for s in states
    ]
})

minimum_duration_actions.to_csv(
    "minimum-duration-actions.tsv",
    sep="\t",
    index=False
)

print(minimum_duration_actions.to_string(index=False))


     state     action
 exposed-1    sleep-8
 exposed-2    sleep-8
 exposed-3    sleep-8
 recovered do-nothing
symptoms-1 do-nothing
symptoms-2 do-nothing
symptoms-3 do-nothing


Submit "minimum-duration-actions.tsv" in Gradescope.

## Part 6: Shorter Twizzleflu?

Compute the expected number of days sick for each state to a file.

In [18]:
# YOUR CHANGES HERE

minimum_duration_values = pd.DataFrame({
    "state": states,
    "expected_duration": [
        max(0.0, -V_duration[s])
        for s in states
    ]
})

print(minimum_duration_values.to_string(index=False))

minimum_duration_values.to_csv(
    "minimum-duration-values.tsv",
    sep="\t",
    index=False
)

print("\nFile saved: minimum-duration-values.tsv")


     state  expected_duration
 exposed-1           1.250000
 exposed-2           2.500000
 exposed-3           5.000000
 recovered           0.000000
symptoms-1          10.000000
symptoms-2           6.666667
symptoms-3           3.333333

File saved: minimum-duration-values.tsv


Save the expected sick days for each state to a file "minimum-duration-days.tsv" with columns state and expected_sick_days.

In [19]:
# YOUR CHANGES HERE

minimum_duration_days = pd.DataFrame({
    "state": states,
    "expected_sick_days": [
        max(0.0, -V_duration[s])
        for s in states
    ]
})

minimum_duration_days.to_csv(
    "minimum-duration-days.tsv",
    sep="\t",
    index=False
)

print(minimum_duration_days.to_string(index=False))
print("\nFile saved: minimum-duration-days.tsv")


     state  expected_sick_days
 exposed-1            1.250000
 exposed-2            2.500000
 exposed-3            5.000000
 recovered            0.000000
symptoms-1           10.000000
symptoms-2            6.666667
symptoms-3            3.333333

File saved: minimum-duration-days.tsv


Submit "minimum-duration-days.tsv" in Gradescope.

## Part 7: Speed vs Pampering

Compute the expected discomfort using the policy to minimize days sick, and compare the results to the expected discomfort when optimizing to minimize discomfort.

Hint: Make sure that you are reporting on expected discomfort, not repeating the expected number of days.

In [20]:
# YOUR CHANGES HERE

original_reward_map = {
    (row.state, row.action): row.reward
    for row in rewards.itertuples(index=False)
}

V_speed_discomfort = {s: 0.0 for s in states}

tolerance = 1e-10
max_iterations = 100000

for iteration in range(max_iterations):
    new_V = V_speed_discomfort.copy()

    for s in states:
       
        a = duration_policy.get(s)

        if a is None:
            continue

        key = (s, a)

        if key not in original_reward_map:
            continue

        future_value = sum(
            p * V_speed_discomfort[next_state]
            for next_state, p in transition_map.get(key, [])
        )

        new_V[s] = (
            original_reward_map[key] + future_value
        )

    delta = max(
        abs(new_V[s] - V_speed_discomfort[s])
        for s in states
    )

    V_speed_discomfort = new_V

    if delta < tolerance:
        break

print(f"Iterations: {iteration + 1}")
print(f"Converged: {delta < tolerance}")

# Compare expected discomfort for both policies
policy_comparison = pd.DataFrame({
    "state": states,
    "speed_discomfort": [
        max(0.0, -V_speed_discomfort[s])
        for s in states
    ],
    "minimize_discomfort": [
        max(0.0, -V_opt[s])
        for s in states
    ]
})


Iterations: 82
Converged: True


Save the results to a file "policy-comparison.tsv" with columns state, speed_discomfort, and minimize_discomfort.

In [21]:
# YOUR CHANGES HERE

policy_comparison.to_csv(
    "policy-comparison.tsv",
    sep="\t",
    index=False
)

print("\nPolicy Comparison:")
print(policy_comparison.to_string(index=False))

print("\nFile saved: policy-comparison.tsv")


Policy Comparison:
     state  speed_discomfort  minimize_discomfort
 exposed-1          0.833333                 0.75
 exposed-2          1.666667                 1.50
 exposed-3          3.333333                 3.00
 recovered          0.000000                 0.00
symptoms-1          6.666667                 6.00
symptoms-2          5.000000                 4.50
symptoms-3          1.666667                 1.50

File saved: policy-comparison.tsv


Submit "policy-comparison.tsv" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.